<a href="https://colab.research.google.com/github/matthewpecsok/IS4490_student_course_files/blob/main/module_04_tools_and_skills_notebook_LAB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---
## Setups: 1 - Ollama and LangChain

Use Google Colab or a local Jupyter environment. If Ollama and the required Python packages are already installed, skip the installation cell. If the preferred model is unavailable, use an instructor-approved tool-capable equivalent and document the substitution in your AI-use disclosure.

In [3]:
# GOOGLE COLAB ONLY: install Ollama and the Python packages.
# Skip this cell in a configured local environment.
!apt-get update -qq
!apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh
!pip install langchain langchain-ollama matplotlib --quiet

W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 92 not upgraded.
Need to get 644 kB of archives.
After this operation, 1,845 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 zstd amd64 1.5.5+dfsg2-2build1.1 [644 kB]
Fetched 644 kB in 1s (595 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 126952 files and directories currently installed.)
Preparing to unpack .../zs

In [4]:
import requests
from bs4 import BeautifulSoup

In [5]:
# GOOGLE COLAB ONLY: start Ollama in the notebook environment.
# Skip this cell if `ollama serve` is already running locally.
import subprocess
import time

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
time.sleep(3)
print("Ollama server start requested.")

Ollama server start requested.


In [6]:
# Pull the model once. This can take several minutes.
!ollama pull gemma4:12b

In [7]:
from pathlib import Path
import sqlite3

from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
import matplotlib.pyplot as plt

MODEL = "gemma4:12b"
llm = ChatOllama(model=MODEL, temperature=0)

OUTPUT_DIR = Path("module4_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"Ready. Using {MODEL} through Ollama and LangChain.")
print(f"Generated files will stay inside: {OUTPUT_DIR.resolve()}")

Ready. Using gemma4:12b through Ollama and LangChain.
Generated files will stay inside: /content/module4_outputs


In [ ]:
# Smoke test: confirm that the model responds before continuing.
response = llm.invoke("In one sentence, explain why a business AI system might need a tool.")
print(response.content)

---
## Setups: 3 - Ask the Model and Show the Tool Trace

This helper exposes the listed tools to the model, executes at most the first requested tool, and prints the model-visible trace. The host application—not the model—executes the Python function.

In [11]:
def ask_model(question, tools, system_prompt=None):
    system_prompt = system_prompt or (
        "You are a business AI assistant. Select the single tool that best matches "
        "the user's request. Call no tool when none is needed. Never claim that a "
        "tool ran unless its result is present."
    )
    tool_map = {candidate.name: candidate for candidate in tools}
    llm_with_tools = llm.bind_tools(tools)
    messages = [SystemMessage(content=system_prompt), HumanMessage(content=question)]
    response = llm_with_tools.invoke(messages)

    print(f"Question: {question}")

    if not response.tool_calls:
        print("Tool called: none")
        print(f"Answer: {response.content}\n")
        return {
            "question": question,
            "tool_called": None,
            "arguments": None,
            "tool_result": None,
            "answer": response.content,
        }

    if len(response.tool_calls) > 1:
        print(f"Notice: the model requested {len(response.tool_calls)} tools; only the first will run.")

    call = response.tool_calls[0]
    tool_obj = tool_map[call["name"]]
    result = tool_obj.invoke(call["args"])

    print(f"Tool called: {call['name']}")
    print(f"Arguments: {call['args']}")
    print(f"Tool result: {result}")

    messages.extend(
        [
            response,
            ToolMessage(
                content=str(result),
                tool_call_id=call["id"],
                name=call["name"],
            ),
        ]
    )
    final = llm_with_tools.invoke(messages)
    print(f"Answer: {final.content}\n")

    return {
        "question": question,
        "tool_called": call["name"],
        "arguments": call["args"],
        "tool_result": result,
        "answer": final.content,
    }

---
# Tool 1: Scrape Webpages

**Business problem:** How to support sales reps in researching online website with AI?

Notice that this tool is simply a function in Python with the **@tool** decorator.

In [9]:
from langchain_core.tools import tool

@tool
def scrape_webpage(url: str) -> str:
    '''Scrapes the text content from a given URL.

    Use this tool to get the visible text content of a webpage when a user asks a question that can be answered by visiting a URL.

    Args:
        url: The URL of the webpage to scrape.
    '''
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)
        soup = BeautifulSoup(response.text, 'html.parser')
        # Remove script and style elements
        for script_or_style in soup(['script', 'style']):
            script_or_style.decompose()
        text = soup.get_text()
        # Break into lines and remove leading/trailing space on each
        lines = (line.strip() for line in text.splitlines())
        # Break multi-hyphenated words into two and remove empty lines
        chunks = (phrase.strip() for line in lines for phrase in line.split("  "))
        # Drop blank lines
        text = '\n'.join(chunk for chunk in chunks if chunk)

        # Limit the output length to avoid overwhelming the model
        max_length = 2000 # Approximately 500 tokens
        if len(text) > max_length:
            return f"success: {text[:max_length]}... (truncated)"
        return f"success: {text}"
    except requests.exceptions.RequestException as e:
        return f"error: Could not retrieve content from {url}. Error: {e}"
    except Exception as e:
        return f"error: An unexpected error occurred while scraping {url}. Error: {e}"


--- ## Resources


In [13]:
# Run both a valid case and a controlled invalid-input case.
industry_success = ask_model(
    "What is on the university of utah homepage",
    [scrape_webpage],
)

# review the trace below.
# a user would just see the "Answer"

Question: What is on the university of utah homepage
Tool called: scrape_webpage
Arguments: {'url': 'https://www.utah.edu'}
Tool result: success: The University of Utah
Skip to content
Search Site:
Powered by
Search Campus:
Powered by
Search Campus
Campus Alert
The University of Utah
Apply
Apply
Office of Admissions
Prospective Students
Request information
Campus Tours
Events
Undergraduate
Graduate
International Students
Asia Campus
New Students
Freshman
Confirm/Deposit
Financial Aid & Scholarships
New Student Orientation
Registrar
Housing
Academic Calendar
Student Life
Student Affairs
Housing
The MUSS
The Union
Student Life Center
Student Leadership & Involvement
SafeU
U Career Success
Student Experience
Parents & Supporters
Academics
Academic Advising
Catalogs, Schedules & Calendar
Colleges & Departments
Majors
Graduate School
Student Success
Libraries
Global U
Research
VP of Research
Arts
Study in the Arts
Art & Art History
Dance
Film & Media Arts
Music
Theatre
Experience the Arts
E

# Tool 2: Summarize Resume

In [15]:
resume_content = """
John Doe\n123 Main St, Anytown USA\njohn.doe@example.com | (555) 123-4567\n\nSummary:\nA highly motivated and results-oriented professional with 5+ years of experience in software development and project management. Proven ability to lead cross-functional teams, deliver complex projects on time and within budget, and implement innovative solutions.\n\nExperience:\nSenior Software Engineer, Tech Solutions Inc. (2020 - Present)\n  - Led development of a new microservices architecture, resulting in a 30% increase in system performance.\n  - Mentored junior developers and conducted code reviews.\n  - Collaborated with product management to define and prioritize features.\n\nSoftware Engineer, Innovate Corp. (2018 - 2020)\n  - Developed and maintained web applications using Python and React.\n  - Implemented RESTful APIs and integrated third-party services.\n  - Participated in agile development sprints.\n\nEducation:\nMaster of Science in Computer Science, University of California, Berkeley (2018)\nBachelor of Science in Computer Science, University of Washington (2016)\n\nSkills:\nProgramming Languages: Python, Java, JavaScript, C++\nFrameworks: Django, Flask, React, Node.js\nDatabases: PostgreSQL, MySQL, MongoDB\nTools: Git, Docker, Kubernetes, AWS
"""

with open(OUTPUT_DIR / "resume.txt", "w") as f:
    f.write(resume_content)

print(f"Resume file created at: {OUTPUT_DIR / 'resume.txt'}")

Resume file created at: module4_outputs/resume.txt


In [16]:
@tool
def summarize_resume(file_path: str) -> str:
    '''Reads a resume from a text file and generates a summary using the LLM.

    Use this tool to provide a concise overview of a person's professional experience and skills.

    Args:
        file_path: The path to the resume text file.
    '''
    try:
        with open(file_path, "r") as f:
            resume_text = f.read()

        # Use the LLM to summarize the resume content
        summary_prompt = f"Please summarize the following resume in 3-4 sentences:\n\n{resume_text}"
        response = llm.invoke(summary_prompt)
        return f"success: {response.content}"
    except FileNotFoundError:
        return f"error: Resume file not found at {file_path}."
    except Exception as e:
        return f"error: An unexpected error occurred while summarizing the resume. Error: {e}"


In [17]:
resume_summary_result = ask_model(
    "Summarize the resume located at module4_outputs/resume.txt",
    [summarize_resume]
)

Question: Summarize the resume located at module4_outputs/resume.txt
Tool called: summarize_resume
Arguments: {'file_path': 'module4_outputs/resume.txt'}
Tool result: success: John Doe is a results-oriented professional with over five years of experience in software development and project management. He currently serves as a Senior Software Engineer, where he has demonstrated leadership by overseeing microservices architecture and mentoring junior team members. He holds a Master of Science in Computer Science from UC Berkeley and possesses a broad technical skill set including Python, React, and cloud technologies like AWS.
Answer: John Doe is a results-oriented professional with over five years of experience in software development and project management. He currently serves as a Senior Software Engineer, where he has demonstrated leadership by overseeing microservices architecture and mentoring junior team members. He holds a Master of Science in Computer Science from UC Berkeley an

# Two tools!

Can the LLM choose the CORRECT tool when it has 2 to choose from?

In [18]:
resume_summary_result = ask_model(
    "Summarize the resume located at module4_outputs/resume.txt",
    [summarize_resume,scrape_webpage]
)

Question: Summarize the resume located at module4_outputs/resume.txt
Tool called: summarize_resume
Arguments: {'file_path': 'module4_outputs/resume.txt'}
Tool result: success: John Doe is a results-oriented professional with over five years of experience in software development and project management. Currently serving as a Senior Software Engineer, he has a proven track record of leading high-impact projects, such as implementing microservices architectures and mentoring junior team members. He possesses a strong technical foundation in languages like Python and Java, along with proficiency in modern frameworks and cloud tools like Docker and AWS. He holds a Master of Science in Computer Science from UC Berkeley.
Answer: John Doe is a results-oriented professional with over five years of experience in software development and project management. Currently a Senior Software Engineer, he has a proven track record of leading high-impact projects, including implementing microservices arch

In [19]:
resume_summary_result = ask_model(
    "What is on the University of Utah Homepage?",
    [summarize_resume,scrape_webpage]
)

Question: What is on the University of Utah Homepage?
Tool called: scrape_webpage
Arguments: {'url': 'https://www.utah.edu'}
Tool result: success: The University of Utah
Skip to content
Search Site:
Powered by
Search Campus:
Powered by
Search Campus
Campus Alert
The University of Utah
Apply
Apply
Office of Admissions
Prospective Students
Request information
Campus Tours
Events
Undergraduate
Graduate
International Students
Asia Campus
New Students
Freshman
Confirm/Deposit
Financial Aid & Scholarships
New Student Orientation
Registrar
Housing
Academic Calendar
Student Life
Student Affairs
Housing
The MUSS
The Union
Student Life Center
Student Leadership & Involvement
SafeU
U Career Success
Student Experience
Parents & Supporters
Academics
Academic Advising
Catalogs, Schedules & Calendar
Colleges & Departments
Majors
Graduate School
Student Success
Libraries
Global U
Research
VP of Research
Arts
Study in the Arts
Art & Art History
Dance
Film & Media Arts
Music
Theatre
Experience the Arts


# Two requests in one user message

For now we will only execute the first request rather than both. you can imagine that later we will want to "Chain" some requests together for more complex workflows.

In [20]:
resume_summary_result = ask_model(
    "What is on the University of Utah Homepage? Also Summarize the resume located at module4_outputs/resume.txt",
    [summarize_resume,scrape_webpage]
)

Question: What is on the University of Utah Homepage? Also Summarize the resume located at module4_outputs/resume.txt
Notice: the model requested 2 tools; only the first will run.
Tool called: scrape_webpage
Arguments: {'url': 'https://www.utah.edu'}
Tool result: success: The University of Utah
Skip to content
Search Site:
Powered by
Search Campus:
Powered by
Search Campus
Campus Alert
The University of Utah
Apply
Apply
Office of Admissions
Prospective Students
Request information
Campus Tours
Events
Undergraduate
Graduate
International Students
Asia Campus
New Students
Freshman
Confirm/Deposit
Financial Aid & Scholarships
New Student Orientation
Registrar
Housing
Academic Calendar
Student Life
Student Affairs
Housing
The MUSS
The Union
Student Life Center
Student Leadership & Involvement
SafeU
U Career Success
Student Experience
Parents & Supporters
Academics
Academic Advising
Catalogs, Schedules & Calendar
Colleges & Departments
Majors
Graduate School
Student Success
Libraries
Global